# Datos masivos
## Clase 1. Introducción
###### Alberto Benavides

In [ ]:
import os, gc, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
def make_dataset_parquet(out_dir="data_parquet_tx", n_rows=5_000_000, n_parts=20, seed=42):
    rng = np.random.default_rng(seed)
    os.makedirs(out_dir, exist_ok=True)

    rows_per_part = n_rows // n_parts
    base_ts = np.datetime64("2024-01-01")

    n_clients = 200_000
    n_products = 50_000
    n_categories = 200

    for i in range(n_parts):
        n = rows_per_part if i < n_parts - 1 else (n_rows - rows_per_part*(n_parts-1))

        ts = base_ts + rng.integers(0, 60*60*24*180, size=n).astype("timedelta64[s]")
        day = pd.to_datetime(ts).floor("D")

        table = pa.table({
            "ts": pa.array(ts),
            "day": pa.array(day),
            "client_id": pa.array(rng.integers(1, n_clients+1, size=n, dtype=np.int32)),
            "product_id": pa.array(rng.integers(1, n_products+1, size=n, dtype=np.int32)),
            "category": pa.array(rng.integers(1, n_categories+1, size=n, dtype=np.int16)),
            "amount": pa.array(rng.gamma(shape=2.0, scale=20.0, size=n).astype(np.float32)),
            "qty": pa.array(rng.integers(1, 6, size=n, dtype=np.int8)),
            "is_refund": pa.array(rng.random(size=n) < 0.02),
        })

        pq.write_table(table, os.path.join(out_dir, f"part_{i:04d}.parquet"), compression="zstd")

    return out_dir

DATA_DIR = make_dataset_parquet(out_dir="data_parquet_tx", n_rows=5_000_000, n_parts=20)
DATA_GLOB = os.path.join(DATA_DIR, "*.parquet")

In [ ]:
parquet_file = pq.ParquetFile("data_parquet_tx/part_0000.parquet")

tabla = parquet_file.read_row_groups([0])
display(tabla)

pyarrow.Table
ts: timestamp[ms]
day: timestamp[ms]
client_id: int32
product_id: int32
category: int16
amount: float
qty: int8
is_refund: bool
----
ts: [[2024-01-17 01:33:50.000,2024-05-19 07:29:24.000,2024-04-27 19:44:56.000,2024-03-19 23:57:17.000,2024-03-18 22:37:32.000,...,2024-05-29 10:00:29.000,2024-04-27 07:45:51.000,2024-04-28 18:57:41.000,2024-01-23 22:18:33.000,2024-01-10 10:25:14.000]]
day: [[2024-01-17 00:00:00.000,2024-05-19 00:00:00.000,2024-04-27 00:00:00.000,2024-03-19 00:00:00.000,2024-03-18 00:00:00.000,...,2024-05-29 00:00:00.000,2024-04-27 00:00:00.000,2024-04-28 00:00:00.000,2024-01-23 00:00:00.000,2024-01-10 00:00:00.000]]
client_id: [[139679,72355,196096,134995,103885,...,131712,108705,114892,117975,134521]]
product_id: [[1555,47258,977,4601,30473,...,40975,30691,23255,2711,12693]]
category: [[156,7,70,44,43,...,181,2,134,84,89]]
amount: [[71.00876,15.369992,81.104515,49.123653,137.67326,...,6.5071187,33.73344,30.512585,14.224495,56.508648]]
qty: [[5,2,3,1,5,...,4

In [ ]:
tabla.to_pandas().head(10)

,ts,day,client_id,product_id,category,amount,qty,is_refund
0,2024-01-17 01:33:50,2024-01-17,139679,1555,156,71.008759,5,False
1,2024-05-19 07:29:24,2024-05-19,72355,47258,7,15.369992,2,False
2,2024-04-27 19:44:56,2024-04-27,196096,977,70,81.104515,3,False
3,2024-03-19 23:57:17,2024-03-19,134995,4601,44,49.123653,1,False
4,2024-03-18 22:37:32,2024-03-18,103885,30473,43,137.673264,5,False
5,2024-06-03 13:08:34,2024-06-03,102647,36500,120,24.099833,5,False
6,2024-01-16 11:17:06,2024-01-16,51964,36698,151,11.671719,1,False
7,2024-05-05 12:37:47,2024-05-05,168582,25270,170,24.921925,3,False
8,2024-02-06 06:20:54,2024-02-06,116531,20591,143,26.144341,5,False
9,2024-01-17 22:50:46,2024-01-17,66055,17451,158,53.485691,3,False


In [ ]:
import polars as pl

t0 = time.perf_counter()

lf = pl.scan_parquet(DATA_GLOB)

res_polars = (
    lf
    .filter((pl.col("amount") > 50) & (pl.col("is_refund") == False))
    .group_by("category")
    .agg([
        pl.col("amount").sum().alias("sum_amount"),
        pl.col("amount").mean().alias("mean_amount"),
        pl.len().alias("n"),
    ])
    .sort("sum_amount", descending=True)
    .collect()
)

t1 = time.perf_counter()
print(res_polars.head(5))
print(f"Polars tiempo: {t1 - t0:.3f} s")

shape: (5, 4)
┌──────────┬─────────────┬─────────────┬──────┐
│ category ┆ sum_amount  ┆ mean_amount ┆ n    │
│ ---      ┆ ---         ┆ ---         ┆ ---  │
│ i16      ┆ f32         ┆ f32         ┆ u32  │
╞══════════╪═════════════╪═════════════╪══════╡
│ 26       ┆ 551062.6875 ┆ 76.134659   ┆ 7238 │
│ 28       ┆ 550648.625  ┆ 75.81559    ┆ 7263 │
│ 69       ┆ 549522.9375 ┆ 76.048012   ┆ 7226 │
│ 37       ┆ 548969.625  ┆ 75.63649    ┆ 7258 │
│ 17       ┆ 548626.0    ┆ 75.495529   ┆ 7267 │
└──────────┴─────────────┴─────────────┴──────┘
Polars tiempo: 0.114 s


In [ ]:
import dask.dataframe as dd

t0 = time.perf_counter()

df = dd.read_parquet(DATA_DIR) # Segmentadoooo
df2 = df[(df["amount"] > 50) & (df["is_refund"] == False)]

res_dask = df2.groupby("category").agg({"amount": ["sum", "mean", "count"]}).compute()
res_dask.columns = ["sum_amount", "mean_amount", "n"]
res_dask = res_dask.sort_values("sum_amount", ascending=False).head(5)

t1 = time.perf_counter()
print(res_dask)
print(f"Dask tiempo: {t1 - t0:.3f} s")

           sum_amount  mean_amount     n
category                                
26        551061.7500    76.134533  7238
28        550648.7500    75.815606  7263
69        549522.7500    76.047986  7226
37        548969.9375    75.636530  7258
17        548626.2500    75.495562  7267
Dask tiempo: 0.390 s


In [ ]:
import pandas as pd
import time
import glob

t0 = time.perf_counter()

files = glob.glob(DATA_GLOB) # Carga de archivos
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) # Almacenamiento en RAM

res_pandas = (
    df[(df["amount"] > 50) & (df["is_refund"] == False)]
    .groupby("category")["amount"]
    .agg(
        sum_amount="sum",
        mean_amount="mean",
        n="count"
    )
    .sort_values("sum_amount", ascending=False)
    .head(5)
)

t1 = time.perf_counter()

print(res_pandas)
print(f"Pandas tiempo: {t1 - t0:.3f} s")

           sum_amount  mean_amount     n
category                                
26        551061.6875    76.134521  7238
28        550648.7500    75.815605  7263
69        549522.7500    76.047989  7226
37        548969.9375    75.636528  7258
17        548626.2500    75.495560  7267
Pandas tiempo: 1.382 s


### Instalación de Spark

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

t0 = time.perf_counter()

spark = (
    SparkSession.builder
    .appName("Ejemplo")
    .getOrCreate()
)

df_s = spark.read.parquet(DATA_DIR)

res_spark = (
    df_s
    .filter((F.col("amount") > 50) & (F.col("is_refund") == F.lit(False)))
    .groupBy("category")
    .agg(
        F.sum("amount").alias("sum_amount"),
        F.avg("amount").alias("mean_amount"),
        F.count(F.lit(1)).alias("n")
    )
    .orderBy(F.col("sum_amount").desc())
)

top_spark = res_spark.limit(5).toPandas()

t1 = time.perf_counter()
print(top_spark)
print(f"PySpark tiempo: {t1 - t0:.3f} s")

   category     sum_amount  mean_amount     n
0        26  551061.715797    76.134528  7238
1        28  550648.747658    75.815606  7263
2        69  549522.731480    76.047984  7226
3        37  548969.923367    75.636528  7258
4        17  548626.247726    75.495562  7267
PySpark tiempo: 3.721 s


![](https://i.pinimg.com/originals/0c/64/9a/0c649a17ec1e5f5ca340248b4ef4e4be.gif)

In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession.builder.appName("Mi primera sesion").getOrCreate()

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Fetched 384 kB in 1s (360 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
54 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspe

In [ ]:
spark

### Lectura de datos

In [ ]:
import requests
path = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
response = requests.get(path)
url_content = response.content

csv_file_name = 'owid-covid-data.csv'
csv_file = open(csv_file_name, 'wb')

csv_file.write(url_content)
csv_file.close()

In [ ]:
df = spark.read.csv('/content/'+csv_file_name, header=True, inferSchema=True)

### Dataframes de PySpark

In [ ]:
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

In [ ]:
#Converting a date column
df.select(F.to_date(df.date).alias('date'))

DataFrame[date: date]

In [ ]:
#Summary stats
df.describe().show()

+-------+--------+-------------+-----------+--------------------+------------------+------------------+------------------+------------------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+------------------+------------------+------------------------+------------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-------------------+------------------+------------------------+----------------------+------------------+-------------------------------+-------------------+------------------+-------------+--------------------+--------------------+-----------------------+--------------------+------------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------

In [ ]:
#DataFrame Filtering
df.filter(df.location == "Mexico").orderBy(F.desc("date")).show()

+--------+-------------+--------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+------------------------------

In [ ]:
#Simple Group by Function
df.groupBy("location").sum("new_cases").orderBy(F.desc("sum(new_cases)")).show(truncate=False)

+-----------------------------+--------------+
|location                     |sum(new_cases)|
+-----------------------------+--------------+
|World                        |775935057     |
|High-income countries        |429044052     |
|Asia                         |301564180     |
|Europe                       |252916868     |
|Upper-middle-income countries|251756125     |
|European Union (27)          |185822587     |
|North America                |124492698     |
|United States                |103436829     |
|China                        |99373219      |
|Lower-middle-income countries|92019711      |
|South America                |68811012      |
|India                        |45041748      |
|France                       |38997490      |
|Germany                      |38437756      |
|Brazil                       |37511921      |
|South Korea                  |34571873      |
|Japan                        |33803572      |
|Italy                        |26781078      |
|United Kingd

### Tarea 1 (10 puntos). Creación y operaciones básicas con PySpark

- Instalar en un entorno local o ejecutar Spark en algún servidor en línea (como Google Colab)
- Elegir un conjunto de datos para trabajar durante el tetramestre, definirlo y explicar por qué se elige
- Cargar el conjunto de datos mediante PySpark (No hace falta cargarlo en el repositorio)
- Usar PySpark para filtrar datos, generar estadísticas descriptivas básicas y realizar algunas operaciones aritméticas entre registros y columnas
- Crear un repositorio público para el curso y publicar en un cuaderno esta primera tarea

